# Synthetic Tanimoto experiments

In [ ]:
from pathlib import Path
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

import gc
import json
import math
import random
import sys
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from threadpoolctl import threadpool_limits
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
import bblean

REPO = next(p for p in [Path.cwd(), Path.cwd()/"classix_tanimoto_manhattan", *Path.cwd().parents] if (p / "src/classix_t.py").is_file())
sys.path[:0] = [str(REPO / "src"), str(REPO / "vendor/bitbirch")]
import classix_t
from classix_t import CLASSIX_T
from spmv import row_popcounts, packed_submatxvec
from bitbirch import bitbirch
classix_t.print = lambda *args, **kwargs: None
classix_t.tqdm = lambda iterable, *args, **kwargs: iterable
bitbirch.set_merge("diameter")

SEED = 101
OUTPUT = REPO / "experiments/experiment_1/_generated/synthetic_notebook"
PLOTS = OUTPUT / "plots"
OUTPUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(exist_ok=True)


## Data and parameters

In [ ]:
def generate_data(num_clusters, pops, d, n, flip_prob=0.1, seed=None):
    assert len(pops) > 1, 'You must provide at least 2 cluster mean populations in order to generate meaningful data'
    if len(pops) != num_clusters:
        try:
            print('The number of mean populations is not equal to the number of clusters. Trying to generate mean populations keeping in mind the minimum and maximum pop counts provided by the user')
            pops = np.linspace(min(pops), max(pops), num_clusters)
        except:
            raise ValueError('The number of cluster centers must be equal to the number of cluster populations')
    size_flag = [pops[x] < d for x in range(len(pops))]
    assert all(size_flag), 'The number of dimensions must be greater than the pop counts of the clusters'
    clustc = []
    data = []
    if seed != None:
        np.random.seed(seed)
        random.seed(seed)
    labels = []
    for i in range(num_clusters):
        pop = pops[i]
        p1 = pop / d
        clustc.append(np.random.choice([0, 1], size=d, p=[1 - p1, p1]))
        data.append(clustc[-1])
        labels.append(i)
        for j in range(n - 1):
            labels.append(i)
            flip_indices = np.random.choice([0, 1], size=d, p=[1 - flip_prob * p1, flip_prob * p1])
            new_sample = np.logical_xor(clustc[-1], flip_indices).astype(np.int32)
            data.append(new_sample)
    return (np.array(data).astype(np.int32), np.array(labels).astype(np.int32))


In [ ]:
METHODS = ["dbscan", "classix_packed", "bitbirch", "bblean", "butina"]
CASES = []

def add_case(study, x, clusters=10, pops=(100, 400), d=1000, n=1000,
             radius=0.4, min_pts=5, eps=0.4, cutoff=0.35):
    case_id = (f"popcount_separation_{x}_n10000" if study == "popcount_separation" else
               f"radius_{x:.1f}".replace(".", "p") if study == "radius" else f"{study}_{x}")
    CASES.append(dict(case_id=case_id, study=study, x=x,
        generator=dict(num_clusters=clusters, pops=list(pops), d=d, n=n, flip_prob=0.3, seed=SEED),
        classix=dict(radius=radius, minPts=min_pts, mergeScale=1/math.sqrt(radius)),
        bitbirch=dict(threshold=round(1-radius, 12), branching_factor=50, merge_criterion="diameter"),
        dbscan=dict(eps=eps, min_samples=5, metric="jaccard", n_jobs=1),
        butina=dict(cutoff=cutoff, reordering=False)))

for total in range(10000, 90001, 10000):
    add_case("sample_count", total, n=total//10, min_pts=40)
for pop in range(50, 501, 50):
    add_case("popcount_separation", pop, pops=(50, pop), radius=0.3)
for clusters in (10, 20, 25, 40, 50, 80, 100):
    add_case("cluster_count", clusters, clusters=clusters, pops=(100, 200), n=10000//clusters)
for dimension in range(500, 5001, 500):
    add_case("dimension", dimension, pops=(100, 100), d=dimension)
for radius in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    add_case("radius", radius, min_pts=40, radius=radius, eps=radius, cutoff=radius)

# -1 is a discarded warmup; nonnegative IDs are measured fits.
SCHEDULES = {}
for case in CASES:
    schedules = {method: [-1, 0, 1, 2] for method in METHODS}
    for method in ("dbscan", "butina"):
        schedules[method] = ([-1, 0, 1, 2] if case["study"] == "sample_count" and case["x"] <= 70000 else
                             [-1, 0] if case["study"] == "sample_count" and case["x"] == 80000 else [0])
    SCHEDULES[case["case_id"]] = schedules

assert len(CASES) == 42
pd.DataFrame([dict(case=c["case_id"], rows=c["generator"]["num_clusters"]*c["generator"]["n"],
                   dimensions=c["generator"]["d"], radius=c["classix"]["radius"],
                   minPts=c["classix"]["minPts"]) for c in CASES])


## Cluster and time

In [ ]:
records = []
assignments = {}
MEMORY_BUDGET = 20 * 1024**3

# Fits run in this kernel. The timed construction/fit/assignment boundaries are unchanged.
for case in CASES:
    X, truth = generate_data(**case["generator"])
    order = np.arange(len(X))
    np.random.shuffle(order)  # Continue the generator's seeded random stream.
    X, truth = np.ascontiguousarray(X[order]), truth[order]
    assert np.all(X.sum(axis=1) > 0)
    schedules = SCHEDULES[case["case_id"]]
    failed = set()

    for repeat in sorted({r for schedule in schedules.values() for r in schedule}):
        offset = max(repeat, 0) % len(METHODS)
        for method in METHODS[offset:] + METHODS[:offset]:
            if repeat not in schedules[method] or method in failed:
                continue
            key = (case["case_id"], method)
            row = dict(case_id=case["case_id"], study=case["study"], x=case["x"],
                       method=method, repeat=repeat, warmup=repeat < 0, status="ok")
            model = distances = clusters = data = fingerprints = None

            # Skip only allocations whose lower bound exceeds the original 20 GiB cap.
            pairs = len(X)*(len(X)-1)//2
            butina_lower_bound = pairs*(sys.getsizeof(0.0)+np.dtype(np.intp).itemsize) + 8*len(X)**2
            if method == "butina" and butina_lower_bound > MEMORY_BUDGET:
                row.update(status="skipped_capacity", seconds=None, ari=None, ami=None,
                           reason="Dense RDKit input/matrix lower bound exceeds 20 GiB.")
                records.append(row)
                failed.add(method)
                print(case["case_id"], method, row["status"], flush=True)
                continue

            if method == "dbscan":
                data = X.copy()
            elif method == "bitbirch":
                data = X.astype(np.int64)
            else:
                data = np.packbits(X, axis=1, bitorder="big" if method == "bblean" else "little")
                if method == "butina":
                    fingerprints = [DataStructs.CreateFromBinaryText(r.tobytes()) for r in data]
            gc.collect()
            try:
                with threadpool_limits(limits=1):
                    started = perf_counter()
                    if method == "classix_packed":
                        model = CLASSIX_T(**case["classix"])
                        model.fit(data)
                        labels = model.labels
                    elif method == "dbscan":
                        model = DBSCAN(**case["dbscan"])
                        model.fit(data)
                        labels = model.labels_
                    elif method == "bitbirch":
                        model = bitbirch.BitBirch(threshold=case["bitbirch"]["threshold"],
                                                 branching_factor=case["bitbirch"]["branching_factor"])
                        model.fit(data)
                        labels = model.get_assignments(len(data))
                    elif method == "bblean":
                        model = bblean.BitBirch(**case["bitbirch"])
                        model.fit(data, input_is_packed=True, n_features=8*data.shape[1])
                        labels = model.get_assignments()
                    else:
                        distances = []
                        for i in range(1, len(fingerprints)):
                            similarities = DataStructs.BulkTanimotoSimilarity(fingerprints[i], fingerprints[:i])
                            distances.extend([1-value for value in similarities])
                        clusters = Butina.ClusterData(distances, len(fingerprints), case["butina"]["cutoff"],
                                                      isDistData=True, reordering=False)
                        labels = np.full(len(data), -1, dtype=np.int64)
                        for cluster_id, members in enumerate(clusters):
                            labels[list(members)] = cluster_id
                    elapsed = perf_counter() - started

                labels = np.asarray(labels, dtype=np.int64).copy()
                assert labels.shape == truth.shape
                if key in assignments:
                    previous = np.load(assignments[key])
                    assert adjusted_rand_score(previous, labels) == 1.0
                    assert np.array_equal(previous < 0, labels < 0)
                elif repeat >= 0:
                    path = OUTPUT / f"{case['case_id']}_{method}_labels.npy"
                    np.save(path, labels)
                    assignments[key] = path
                row.update(seconds=elapsed, ari=float(adjusted_rand_score(truth, labels)),
                           ami=float(adjusted_mutual_info_score(truth, labels, average_method="arithmetic")))
                print(case["case_id"], method, repeat, f"{elapsed:.3f}s", f"ARI={row['ari']:.6f}", flush=True)
            except (MemoryError, RuntimeError, ValueError) as error:
                row.update(status="fit_error", seconds=None, ari=None, ami=None, reason=str(error))
                failed.add(method)
                print(case["case_id"], method, row["status"], str(error), flush=True)
            records.append(row)
            pd.DataFrame(records).to_csv(OUTPUT/"fits.csv", index=False)
            model = distances = clusters = data = fingerprints = None
            gc.collect()
    del X, truth

fits = pd.DataFrame(records)
fits.to_csv(OUTPUT/"fits.csv", index=False)


In [ ]:
summary_rows = []
for case in CASES:
    for method in METHODS:
        rows = fits[(fits.case_id == case["case_id"]) & (fits.method == method)]
        expected = SCHEDULES[case["case_id"]][method]
        complete = sorted(rows.repeat.tolist()) == expected and rows.status.eq("ok").all()
        measured = rows[(rows.repeat >= 0) & rows.status.eq("ok")]
        summary_rows.append(dict(case_id=case["case_id"], study=case["study"], x=case["x"], method=method,
            status="ok" if complete else (rows.iloc[-1].status if len(rows) else "not_run"),
            runtime=float(measured.seconds.median()) if complete else np.nan,
            ari=float(measured.ari.median()) if complete else np.nan,
            ami=float(measured.ami.median()) if complete else np.nan))
results = pd.DataFrame(summary_rows)
results.to_csv(OUTPUT/"summary.csv", index=False)
results


## Silhouette

In [ ]:
def tanimoto_silhouette(packed, labels, sample_size=2000, seed=101):
    keep = labels >= 0
    names, inverse, counts = np.unique(labels[keep], return_inverse=True, return_counts=True)
    n = int(keep.sum())
    if not 2 <= len(names) <= n-1:
        return np.nan, np.nan
    compact = np.full(len(labels), -1, dtype=np.int32)
    compact[keep] = inverse
    priority = np.random.default_rng(seed).permutation(len(labels))
    queries = np.sort(priority[keep[priority]][:min(sample_size, n)])
    populations = row_popcounts(packed)
    values = []
    for q in queries:
        group = compact[q]
        if counts[group] == 1:
            values.append(0.0)
            continue
        intersections = packed_submatxvec(packed, 0, len(packed), packed[q])
        unions = populations + populations[q] - intersections
        distances = np.divide(unions-intersections, unions, out=np.zeros(len(packed), dtype=np.float64), where=unions != 0)
        sums = np.bincount(compact[keep], weights=distances[keep], minlength=len(counts))
        a = sums[group]/(counts[group]-1)
        means = sums/counts
        means[group] = np.inf
        b = float(means.min())
        values.append((b-a)/max(a,b) if max(a,b) else 0.0)
    values = np.asarray(values)
    se = float(values.std(ddof=1)/np.sqrt(len(values))*np.sqrt(1-len(values)/n)) if len(values)>1 else 0.0
    return float(values.mean()), se


In [ ]:
results["silhouette"] = np.nan
results["silhouette_se"] = np.nan
for case in CASES:
    X, truth = generate_data(**case["generator"])
    order = np.arange(len(X))
    np.random.shuffle(order)
    packed = np.packbits(np.ascontiguousarray(X[order]), axis=1, bitorder="little")
    for method in METHODS:
        row = results.index[(results.case_id == case["case_id"]) & (results.method == method)][0]
        if results.loc[row, "status"] != "ok":
            continue
        labels = np.load(assignments[(case["case_id"], method)], allow_pickle=False)
        value, se = tanimoto_silhouette(packed, labels, seed=SEED)
        results.loc[row, ["silhouette", "silhouette_se"]] = [value, se]
    results.to_csv(OUTPUT/"summary.csv", index=False)
    del X, packed, truth
results


## Plots

In [ ]:
STYLES = {
    "classix_packed": ("CLASSIX_T", "red", "x", "--"),
    "dbscan": ("DBSCAN", "blue", "o", "-."),
    "bitbirch": ("BitBIRCH", "green", "v", ":"),
    "bblean": ("BitBIRCH-Lean", "#9467bd", "s", "--"),
    "butina": ("Taylor-Butina", "#e68613", "D", "-"),
}
STUDIES = (
    ("sample_count", "Scaling with number of data points", "Number of points", "log"),
    ("popcount_separation", "Scaling with separability of data", "Max pop of seed vector", "log"),
    ("cluster_count", "Scaling with number of clusters", "Number of clusters", "linear"),
    ("dimension", "Scaling with dimensionality", "Number of dimensions", "linear"),
    ("radius", "Sensitivity to Tanimoto radius", "Tanimoto radius r", "log"),
)
plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":10, "axes.titlesize":12,
    "axes.labelsize":10, "xtick.labelsize":10, "ytick.labelsize":10,
    "axes.spines.top":True, "axes.spines.right":True, "axes.grid":False,
    "figure.facecolor":"white", "axes.facecolor":"white", "svg.fonttype":"none", "pdf.fonttype":42})

for study, title, xlabel, time_scale in STUDIES:
    subset = results[results.study == study]
    if subset.empty:
        continue
    xs = np.array(sorted(subset.x.unique()), dtype=float)
    for metric, ylabel in (("runtime", "Time taken (in seconds)"), ("ari", "Adjusted Rand index"),
                           ("ami", "Adjusted mutual information"), ("silhouette", "Tanimoto silhouette")):
        fig, ax = plt.subplots(figsize=(6.4,4.0))
        fig.subplots_adjust(left=.14, right=.97, top=.90, bottom=.15)
        handles, finite_values = [], []
        for method, (label, color, marker, linestyle) in STYLES.items():
            rows = subset[subset.method == method].set_index("x").reindex(xs)
            ys = rows[metric].to_numpy(dtype=float)
            finite_values.extend(ys[np.isfinite(ys)].tolist())
            ax.plot(xs, ys, color=color, linestyle=linestyle, marker=marker, linewidth=1.5, markersize=6)
            handles.append(Line2D([0],[0],label=label,color=color,linestyle=linestyle,marker=marker,linewidth=1.5,markersize=6))
            if metric == "silhouette":
                width = 1.96*rows.silhouette_se.to_numpy(dtype=float)
                ax.fill_between(xs, ys-width, ys+width, color=color, alpha=.12, linewidth=0)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        span = float(xs[-1]-xs[0])
        padding = 4000 if study=="sample_count" else 20 if study=="popcount_separation" else .05*span if span else max(1,abs(float(xs[0]))*.05)
        ax.set_xlim(float(xs[0])-padding, float(xs[-1])+padding)
        ax.set_xticks(xs)
        ax.tick_params(axis="x",labelsize=10 if study=="popcount_separation" else 8 if study=="sample_count" else 9)
        if metric == "runtime":
            ax.set_yscale(time_scale)
            if not finite_values:
                ax.set_ylim((.01,10) if time_scale=="log" else (0,1))
            elif time_scale == "linear":
                ax.set_ylim(bottom=0)
        else:
            ax.set_ylim(min(-.05,min(finite_values,default=0)-.05),1.055)
        ax.legend(handles=handles,loc="best" if study=="sample_count" else "upper left",fontsize=8,frameon=True)
        if study=="popcount_separation" and metric=="runtime" and finite_values:
            fig.canvas.draw()
            lower=ax.get_ylim()[0]
            box=ax.get_legend().get_window_extent(fig.canvas.get_renderer())
            fraction=(box.y0-6*fig.dpi/100-ax.bbox.y0)/ax.bbox.height
            needed=lower*(max(finite_values)/lower)**(1/fraction)
            unit=10**math.floor(math.log10(needed))
            ax.set_ylim(top=math.ceil(needed/unit)*unit)
        for extension in ("png","pdf","svg"):
            fig.savefig(PLOTS/f"{study}_{metric}.{extension}",dpi=300,facecolor="white")
        plt.show()
        plt.close(fig)
